# CMSC 173 &middot; Machine Learning &mdash; Week 12 Lab
## Clustering: K-Means from Scratch

Everything so far had *labels*. Clustering is **unsupervised** &mdash; the data comes with no answers,
and we look for natural groups anyway (customer segments, barangay types, image colours). You'll
build **k-means** from scratch &mdash; the assign-then-update loop &mdash; watch it converge, and use the
**elbow method** to guess how many clusters there are.

**How this lab works.** Each part = a short **plain-English explainer**, a **code cell**
you run, a **line-by-line walkthrough** of what it did, and an **Answer here** box. The
code does the maths; we *graph* the results so you can see what is going on.

**NumPy + Matplotlib (from scratch).** **Not graded.** About 55 minutes.

---
## Part 0 &middot; Setup + unlabelled data

We make blobs but then **throw the labels away** &mdash; the whole point is to recover the groups
without them.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(173)

true_centres = np.array([[0,0], [5,5], [0,6], [6,1]])
X = np.vstack([rng.normal(c, 0.7, (60,2)) for c in true_centres])   # 4 blobs, 240 points
rng.shuffle(X)

plt.figure(figsize=(6,5)); plt.scatter(X[:,0], X[:,1], alpha=0.6, color='gray')
plt.title('Unlabelled data: how many groups do you see?'); plt.tight_layout(); plt.show()

**Reading the code:** four Gaussian blobs, stacked and shuffled, then plotted in a single grey
colour &mdash; because a clustering algorithm doesn't get to see the groups. Your eye probably counts
four; let's make the computer find them.

---
## Part 1 &middot; K-means from scratch

K-means repeats two steps until nothing changes:
1. **Assign** each point to the nearest centre.
2. **Update** each centre to the mean of the points assigned to it.

That's it. Start with random centres and let it settle.

In [ ]:
def one_run(X, k, iters=20):
    centres = X[rng.choice(len(X), k, replace=False)]     # (1) start: k random points as centres
    for _ in range(iters):
        dists = np.linalg.norm(X[:,None,:] - centres[None,:,:], axis=2)  # (2) every point-to-centre distance
        labels = dists.argmin(axis=1)                     # (3) ASSIGN: nearest centre
        new = np.array([X[labels==j].mean(axis=0) if np.any(labels==j) else centres[j]
                        for j in range(k)])               # (4) UPDATE: mean of each group
        if np.allclose(new, centres): break               # (5) stop if centres stopped moving
        centres = new
    inertia = sum(((X[labels==j]-centres[j])**2).sum() for j in range(k))  # (6) tightness score
    return labels, centres, inertia

def kmeans(X, k, n_init=8):
    runs = [one_run(X, k) for _ in range(n_init)]         # (7) several random starts...
    return min(runs, key=lambda r: r[2])                  # (8) ...keep the TIGHTEST (lowest inertia)

labels, centres, inertia = kmeans(X, k=4)
print(f'inertia (total within-cluster spread) = {inertia:.1f}')

plt.figure(figsize=(6,5))
plt.scatter(X[:,0], X[:,1], c=labels, cmap='viridis', alpha=0.6)
plt.scatter(centres[:,0], centres[:,1], c='red', marker='X', s=200, label='centres')
plt.title('k-means found 4 clusters'); plt.legend(); plt.tight_layout(); plt.show()

**Reading the code, line by line:**
- **(1)** pick `k` random data points as the starting centres.
- **(2)** the broadcasting trick computes the distance from *every* point to *every* centre at once.
- **(3)** **assign**: each point takes the label of its nearest centre.
- **(4)** **update**: each centre jumps to the average position of its assigned points.
- **(5)** if the centres stop moving, we've converged &mdash; stop early.
- **(6)** `inertia` = total squared distance of points to their centre; lower = tighter clusters.
- **(7)&ndash;(8)** a single random start can settle into a *bad* arrangement, so `kmeans` runs the whole
  loop several times from different starts and keeps the **tightest** result. The plot shows four
  coloured groups with red centres in the middle of each &mdash; recovered without ever seeing a label.

**Answer here:**

1. `kmeans` runs `one_run` eight times and keeps the lowest-inertia result. Why not just one start?
   (Try calling `one_run(X, 4)` a few times and watch its inertia jump around.)
   &rarr; *your answer*

---
## Part 2 &middot; But how many clusters? The elbow method

We told it `k=4`. In real data you don't know `k`. Trick: run k-means for many `k` and plot the
**inertia**. It always falls as `k` grows (more centres = tighter), but there's usually a **kink**
&mdash; an 'elbow' &mdash; where extra clusters stop helping much. That kink is a good guess for `k`.

In [ ]:
ks = range(1, 9)
inertias = [kmeans(X, k)[2] for k in ks]

plt.figure(figsize=(7,4))
plt.plot(list(ks), inertias, 'o-')
plt.axvline(4, ls='--', color='gray', label='elbow ~ 4')
plt.xlabel('number of clusters k'); plt.ylabel('inertia (total spread)')
plt.title('Elbow method: the kink suggests k'); plt.legend(); plt.tight_layout(); plt.show()
print('inertia per k:', [round(v) for v in inertias])

**Reading the code:** we cluster for `k = 1 … 8` and record each inertia. The curve drops steeply up
to `k=4`, then flattens &mdash; adding a 5th cluster barely tightens anything. That bend is the **elbow**,
and it points at the 4 groups we built. It's a guide, not a law: real elbows can be fuzzy.

**Answer here:**

1. Inertia keeps dropping even after `k=4`. Why would 'just pick the k with the lowest inertia'
   be a bad rule (what does `k = number of points` give you)?
   &rarr; *your answer*

---
## Part 3 &middot; Sanity check against sklearn

You built the loop; sklearn wraps it (with smarter initialisation). Compare the inertias for k=4.

In [ ]:
from sklearn.cluster import KMeans
km = KMeans(n_clusters=4, n_init=10, random_state=0).fit(X)
print(f'sklearn inertia   = {km.inertia_:.1f}')
print(f'your inertia (k=4)= {kmeans(X, 4)[2]:.1f}')

**Reading the code:** `KMeans` runs the same assign/update idea. Like your `kmeans`, `n_init=10` tries
several random starts and keeps the best; it also uses a smarter first guess (**k-means++**) instead
of pure random. The inertias come out essentially the same &mdash; you rebuilt a real library.

**Answer here:**

1. Both your `kmeans` and sklearn use several random starts. In one sentence, what bad outcome does
   trying multiple starts protect against?
   &rarr; *your answer*

---
## Where you actually are

Set the pace honestly. Replace each `-` with: **solid** / **rusty** / **never really got it**.

| | You |
|---|---|
| What 'unsupervised' means | - |
| The k-means assign/update loop | - |
| What inertia measures | - |
| Reading an elbow plot | - |
| Why random starts can matter | - |

**Which part took longest, and where did you get stuck?**
&rarr; *your answer*

**In one plain sentence: what is k-means actually trying to minimise?**
&rarr; *your answer*

---
## Stretch &mdash; optional

Required part is done; nothing below is graded.

### Stretch &middot; A bad k

Cluster the data with `k=2` and plot it. How does k-means *force* four natural groups into two?
Fill it in.

In [ ]:
# your code here: labels2, centres2, _ = kmeans(X, 2); scatter X coloured by labels2


---
## Submitting

Run the cell below. It uploads this notebook straight from Colab &mdash; nothing to download.

You need a **submit token**: open
[https://portal.latarak.com/student/submit-token](https://portal.latarak.com/student/submit-token),
sign in, press the button, then paste it when the cell asks. The cell hides what you type.

In [ ]:
# --- Submit this notebook ------------------------------------------------------
# Colab only. Anywhere else, use the manual route described below this cell.
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, WEEK = "https://portal.latarak.com", "cmsc173", 12

try:
    from google.colab import _message
except ImportError:
    raise SystemExit(
        "Not running in Colab. Download this notebook "
        "(File > Download > Download .ipynb) and upload it at "
        "https://portal.latarak.com/course/cmsc173/lab/12/submit"
    )

nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]
token = getpass.getpass("Submit token (hidden as you type): ").strip()

req = urllib.request.Request(
    PORTAL + "/api/labs/" + COURSE + "/submit-notebook",
    data=json.dumps({"week": WEEK, "notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["course"], "week", out["week"], "for", out["student"])
    print(out["cells"], "cells,", out["executed"], "executed")
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))

Prefer to do it by hand? **File &rarr; Download &rarr; Download .ipynb**, then go to the
[Week 12 submission page](https://portal.latarak.com/course/cmsc173/lab/12/submit) and upload it.

Blank cells are fine and guesses are fine. Don't polish this until it hides what you knew.